# 02 - EDA: Web Scraping Pré-processado

Exploração dos dados coletados via web scraping (~570k linhas, 20MB).

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

con = duckdb.connect()
SCRAPING = "'../data/web_scraping_pre_processado/*.snappy.parquet'"

In [ ]:
# Visão geral
con.execute(f"DESCRIBE SELECT * FROM read_parquet({SCRAPING}) LIMIT 1").df()

In [ ]:
con.execute(f"SELECT COUNT(*) as total FROM read_parquet({SCRAPING})").df()

In [ ]:
# Amostra
con.execute(f"SELECT * FROM read_parquet({SCRAPING}) LIMIT 10").df()

In [ ]:
# Valores nulos por coluna
con.execute(f"""
    SELECT
        COUNT(*) FILTER (WHERE WEBSCRAPING_TIPO_ESTABELECIMENTO IS NULL) AS tipo_nulos,
        COUNT(*) FILTER (WHERE WEBSCRAPING_UF IS NULL) AS uf_nulos,
        COUNT(*) FILTER (WHERE WEBSCRAPING_MUNICIPIO IS NULL) AS municipio_nulos,
        COUNT(*) FILTER (WHERE WEBSCRAPING_LOGRADOURO IS NULL) AS logradouro_nulos,
        COUNT(*) FILTER (WHERE WEBSCRAPING_CEP IS NULL) AS cep_nulos
    FROM read_parquet({SCRAPING})
""").df()

In [ ]:
# Distribuição por tipo de estabelecimento
df_tipo = con.execute(f"""
    SELECT WEBSCRAPING_TIPO_ESTABELECIMENTO as tipo, COUNT(*) as total
    FROM read_parquet({SCRAPING})
    GROUP BY tipo
    ORDER BY total DESC
    LIMIT 20
""").df()

plt.figure(figsize=(12, 6))
sns.barplot(data=df_tipo, x='total', y='tipo')
plt.title('Top 20 tipos de estabelecimento')
plt.tight_layout()
plt.savefig('../outputs/figures/scraping_por_tipo.png', dpi=150)
plt.show()

In [ ]:
# Distribuição por UF
con.execute(f"""
    SELECT WEBSCRAPING_UF as uf, COUNT(*) as total
    FROM read_parquet({SCRAPING})
    GROUP BY uf
    ORDER BY total DESC
""").df()